In [14]:
#!/usr/bin/env python3
"""
Fit DDM and SRDM to rr98 WITHOUT a lapse mixture.

Instead of a lapse process, trials are trimmed (fastest/slowest 1% per
participant x instruction x difficulty cell removed, Ratcliff 2008 style)
and t0 is hard-bounded above by the fastest surviving RT for that
participant (t0_hi = min(RT) over the trimmed data). No blend, no
p_lapse, no mixture -- the model likelihood is evaluated directly.

Uses physical-brightness correctness (strength > 16 = bright).
Excludes strength == 16 (exactly ambiguous), same as the lapse version.

OUTPUT FILES (one per model, all participants stacked, same convention
as the lapse-model fitting cell):
  - fits_ddm_nolapse.csv
  - fits_srdm_nolapse.csv
"""

import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

from datetime import datetime, timezone

import numpy as np
import pandas as pd
from cmdstanpy import CmdStanModel

# ╔═══════════════════════════════════════════════════════════╗
# ║  CONFIGURATION — edit these                              ║
# ╚═══════════════════════════════════════════════════════════╝
for k in [16]:
    N_LEVELS      = k      # <-- CHANGE THIS: 7, 11, 16, 33, etc. (16 was the
                            #     BIC-best resolution found with the lapse models)
    DATA_PATH     = "../../rr98.csv"
    STAN_DDM      = "DDM_rr98_nolapse_st0_final.stan"

    PARTICIPANTS  = ["jf", "kr", "nh"]
    TRIM_LOW      = 0.01    # trim fastest 1%
    TRIM_HIGH     = 0.99    # trim slowest 1% (i.e. keep [1st, 99th] percentile)

    DDM_OUT  = "fits_ddm_nolapseSt0.csv"



    # ═══════════════════════════════════════════════════════════
    # Data prep
    # ═══════════════════════════════════════════════════════════
    def load_data():
        df = pd.read_csv(DATA_PATH)
        df = df[df["outlier"] == False].copy()
        df["correct"] = df["correct"].astype(int)
        df["sat_id"] = df["instruction"].map({"speed": 1, "accuracy": 2})

        # Physical correctness
        df = df[df["strength"] != 16].copy()
        df["act_correct"] = ((df["strength"] > 16) == (df["response"] == "light")).astype(int)

        # Bin strength into N_LEVELS groups (identical logic to the lapse version)
        if N_LEVELS == 33:
            unique_strengths = sorted(df["strength"].unique())
            strength_to_level = {s: i+1 for i, s in enumerate(unique_strengths)}
            df["diff_level"] = df["strength"].map(strength_to_level)
            actual_levels = len(unique_strengths)
        else:
            def _qcut_levels(s):
                return pd.qcut(s, q=N_LEVELS, labels=False, duplicates="drop") + 1
            df["diff_level"] = df.groupby("id")["strength"].transform(_qcut_levels)
            actual_levels = df["diff_level"].nunique()

        df["cell"] = (df["sat_id"] - 1) * actual_levels + df["diff_level"]

        print(f"N_LEVELS requested: {N_LEVELS}, actual unique levels: {actual_levels}")
        print(f"Trials before trimming: {len(df)}")

        return df, actual_levels


    def trim_extremes(df, low=TRIM_LOW, high=TRIM_HIGH):
        """Trim the fastest/slowest tails within each (id, cell) group, i.e.
        per participant x instruction x difficulty cell -- not globally."""
        def _trim_group(g):
            lo, hi = g["rt"].quantile([low, high])
            return g[(g["rt"] >= lo) & (g["rt"] <= hi)]

        trimmed = df.groupby(["id", "cell"], group_keys=False).apply(_trim_group)
        print(f"Trials after trimming ({low:.0%}/{high:.0%} per id x cell): "
            f"{len(trimmed)}  (dropped {len(df) - len(trimmed)}, "
            f"{(1 - len(trimmed)/len(df)):.1%})")
        return trimmed


    def build_data(df, pid, n_levels):
        d = df[df["id"] == pid]
        d_correct = d[d["act_correct"] == 1]
        d_false = d[d["act_correct"] == 0]
        t0_hi = float(d["rt"].min())
        return {
            "N_LEVELS": n_levels,
            "N_correct": len(d_correct), "N_false": len(d_false),
            "rt_correct": d_correct["rt"].to_numpy(),
            "rt_false": d_false["rt"].to_numpy(),
            "cell_correct": d_correct["cell"].to_numpy(dtype=int),
            "cell_false": d_false["cell"].to_numpy(dtype=int),
            "t0_hi": t0_hi,
            "precision": 0.1,   # <-- ADD THIS
        }


    # ═══════════════════════════════════════════════════════════
    # Fitting
    # ═══════════════════════════════════════════════════════════
    def fit_ddm(model, data):
        nl = data["N_LEVELS"]
        st0_init = 0.03
        t0_target = 0.2 * data["t0_hi"]
        inits = {
            "a": [0.8, 1.5], "v_base": [2.0]*nl,
            "sv": 0.5, "sz": 0.1,
            "t0_prop": min(0.95, t0_target / (data["t0_hi"] - st0_init/2)),
            "st0": st0_init,
        }
        return model.optimize(data=data, inits=inits, algorithm="lbfgs",
                            iter=500, show_console=True)





    def aic_bic(mle, n_params):
        p = mle.optimized_params_pd
        ll_cols = [c for c in p.columns if c.startswith("log_lik")]
        total_ll = p[ll_cols].iloc[0].sum()
        n = len(ll_cols)
        return {"n_params": n_params, "n_trials": n, "log_lik": total_ll,
                "AIC": 2*n_params - 2*total_ll,
                "BIC": n_params*np.log(n) - 2*total_ll}


    # ═══════════════════════════════════════════════════════════
    # Consolidated CSV output (same convention as the lapse fitting cell)
    # ═══════════════════════════════════════════════════════════
    def save_fit_row(csv_path, pid, n_levels_requested, n_levels_actual, mle, ic,
                    extra=None):
        raw = mle.optimized_params_pd.iloc[0]
        keep_cols = [c for c in raw.index if not c.startswith("log_lik")]
        row = raw[keep_cols].to_dict()

        row = {
            "pid": pid,
            "n_levels_requested": n_levels_requested,
            "n_levels_actual": n_levels_actual,
            "n_params": ic["n_params"],
            "n_trials": ic["n_trials"],
            "log_lik_total": ic["log_lik"],
            "AIC": ic["AIC"],
            "BIC": ic["BIC"],
            "fit_time_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            **(extra or {}),
            **row,
        }
        new_row = pd.DataFrame([row])

        if os.path.exists(csv_path):
            existing = pd.read_csv(csv_path)
            mask_same = (existing["pid"] == pid) & (existing["n_levels_actual"] == n_levels_actual)
            existing = existing[~mask_same]
            combined = pd.concat([existing, new_row], ignore_index=True, sort=False)
        else:
            combined = new_row

        combined = combined.sort_values(["n_levels_actual", "pid"]).reset_index(drop=True)
        combined.to_csv(csv_path, index=False)
        return combined


    # ═══════════════════════════════════════════════════════════
    # Main
    # ═══════════════════════════════════════════════════════════
    def main():
        df, actual_levels = load_data()
        df = trim_extremes(df)

        # Parameter counts (no p_lapse in the no-lapse models):
        #   DDM:  a[2] + v_base[levels] + sv + sz + t0        = 2 + levels + 3
        #   SRDM: c[2] + d_base[levels] + B + r + t0          = 2 + levels + 3
        ddm_n_params  = 2 + actual_levels + 3


        print(f"\nDDM params: {ddm_n_params} ({actual_levels} drift rates, no lapse)")

        print("\nCompiling models...")
        ddm_model = CmdStanModel(stan_file=STAN_DDM)

        for pid in PARTICIPANTS:
            print(f"\n{'='*60}")
            print(f"  {pid}  (N_LEVELS={actual_levels}, no lapse, trimmed)")
            print(f"{'='*60}")

            data = build_data(df, pid, actual_levels)
            print(f"  N_correct={data['N_correct']}  N_false={data['N_false']}  "
                f"t0_hi (min trimmed RT)={data['t0_hi']:.4f}")

            # DDM
            print(f"\n  --- DDM (no lapse) ---")
            ddm_mle = fit_ddm(ddm_model, data)
            ic = aic_bic(ddm_mle, ddm_n_params)
            save_fit_row(DDM_OUT, pid, N_LEVELS, actual_levels, ddm_mle, ic,
                        extra={"t0_hi": data["t0_hi"]})
            row = ddm_mle.optimized_params_pd.iloc[0]
            print(f"  a=[{row['a[1]']:.3f}, {row['a[2]']:.3f}]  t0={row['t0']:.4f}  "
                f"sv={row['sv']:.4f}  sz={row['sz']:.4f}   st0={row['st0']:.4f}")
            print(f"  LL={ic['log_lik']:.1f}  AIC={ic['AIC']:.1f}  BIC={ic['BIC']:.1f}")



            

        print(f"\nAll fits written/updated in:\n  {DDM_OUT}\n  ")


    if __name__ == "__main__":
        main()


/var/folders/g5/d_krhx3n0xg4nf8x8frstwdw0000gq/T/ipykernel_1342/454201421.py:86: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  trimmed = df.groupby(["id", "cell"], group_keys=False).apply(_trim_group)
13:15:38 - cmdstanpy - INFO - Chain [1] start processing


N_LEVELS requested: 16, actual unique levels: 16
Trials before trimming: 22584
Trials after trimming (1%/99% per id x cell): 22060  (dropped 524, 2.3%)

DDM params: 21 (16 drift rates, no lapse)

Compiling models...

  jf  (N_LEVELS=16, no lapse, trimmed)
  N_correct=6264  N_false=894  t0_hi (min trimmed RT)=0.2080

  --- DDM (no lapse) ---
Chain [1] method = optimize
Chain [1] optimize
Chain [1] algorithm = lbfgs (Default)
Chain [1] lbfgs
Chain [1] init_alpha = 0.001 (Default)
Chain [1] tol_obj = 1e-12 (Default)
Chain [1] tol_rel_obj = 10000 (Default)
Chain [1] tol_grad = 1e-08 (Default)
Chain [1] tol_rel_grad = 1e+07 (Default)
Chain [1] tol_param = 1e-08 (Default)
Chain [1] history_size = 5 (Default)
Chain [1] jacobian = false (Default)
Chain [1] iter = 500
Chain [1] save_iterations = false (Default)
Chain [1] id = 1 (Default)
Chain [1] data
Chain [1] file = /var/folders/g5/d_krhx3n0xg4nf8x8frstwdw0000gq/T/tmpynjlnw0c/oe_7ob3j.json
Chain [1] init = /var/folders/g5/d_krhx3n0xg4nf8x8fr

13:19:45 - cmdstanpy - INFO - Chain [1] done processing
13:19:45 - cmdstanpy - INFO - Chain [1] start processing


Chain [1] Optimization terminated normally:
Chain [1] Convergence detected: relative gradient magnitude is below tolerance
Chain [1] 
  a=[0.815, 2.295]  t0=0.1851  sv=0.6338  sz=0.0668   st0=0.0457
  LL=2677.3  AIC=-5312.6  BIC=-5168.2

  kr  (N_LEVELS=16, no lapse, trimmed)
  N_correct=6089  N_false=934  t0_hi (min trimmed RT)=0.2060

  --- DDM (no lapse) ---
Chain [1] method = optimize
Chain [1] optimize
Chain [1] algorithm = lbfgs (Default)
Chain [1] lbfgs
Chain [1] init_alpha = 0.001 (Default)
Chain [1] tol_obj = 1e-12 (Default)
Chain [1] tol_rel_obj = 10000 (Default)
Chain [1] tol_grad = 1e-08 (Default)
Chain [1] tol_rel_grad = 1e+07 (Default)
Chain [1] tol_param = 1e-08 (Default)
Chain [1] history_size = 5 (Default)
Chain [1] jacobian = false (Default)
Chain [1] iter = 500
Chain [1] save_iterations = false (Default)
Chain [1] id = 1 (Default)
Chain [1] data
Chain [1] file = /var/folders/g5/d_krhx3n0xg4nf8x8frstwdw0000gq/T/tmpynjlnw0c/iu93beqn.json
Chain [1] init = /var/folders/g

13:24:08 - cmdstanpy - INFO - Chain [1] done processing
13:24:09 - cmdstanpy - INFO - Chain [1] start processing


Chain [1] Optimization terminated normally:
Chain [1] Convergence detected: relative gradient magnitude is below tolerance
Chain [1] 
  a=[0.785, 2.279]  t0=0.1824  sv=0.8523  sz=0.0646   st0=0.0473
  LL=3182.5  AIC=-6323.0  BIC=-6179.0

  nh  (N_LEVELS=16, no lapse, trimmed)
  N_correct=7251  N_false=628  t0_hi (min trimmed RT)=0.2260

  --- DDM (no lapse) ---
Chain [1] method = optimize
Chain [1] optimize
Chain [1] algorithm = lbfgs (Default)
Chain [1] lbfgs
Chain [1] init_alpha = 0.001 (Default)
Chain [1] tol_obj = 1e-12 (Default)
Chain [1] tol_rel_obj = 10000 (Default)
Chain [1] tol_grad = 1e-08 (Default)
Chain [1] tol_rel_grad = 1e+07 (Default)
Chain [1] tol_param = 1e-08 (Default)
Chain [1] history_size = 5 (Default)
Chain [1] jacobian = false (Default)
Chain [1] iter = 500
Chain [1] save_iterations = false (Default)
Chain [1] id = 1 (Default)
Chain [1] data
Chain [1] file = /var/folders/g5/d_krhx3n0xg4nf8x8frstwdw0000gq/T/tmpynjlnw0c/2hezdpf6.json
Chain [1] init = /var/folders/g

13:27:23 - cmdstanpy - INFO - Chain [1] done processing


Chain [1] 329       4936.46   2.31055e-05      0.193147    0.000126       0.001      401  LS failed, Hessian reset
Chain [1] Optimization terminated normally:
Chain [1] Convergence detected: relative gradient magnitude is below tolerance
Chain [1] 
  a=[0.997, 1.889]  t0=0.2038  sv=0.7938  sz=0.0864   st0=0.0444
  LL=4967.6  AIC=-9893.1  BIC=-9746.7

All fits written/updated in:
  fits_ddm_nolapseSt0.csv
  


In [ ]:
fit_ddm = pd.read_csv("fits_ddm_nolapseSt0.csv")
print(fit_ddm["log_lik_total"])
fit_ddm


0    2677.297072
1    3182.504234
2    4967.571920
Name: log_lik_total, dtype: float64


,pid,n_levels_requested,n_levels_actual,n_params,n_trials,log_lik_total,AIC,BIC,fit_time_utc,t0_hi,...,v_full[23],v_full[24],v_full[25],v_full[26],v_full[27],v_full[28],v_full[29],v_full[30],v_full[31],v_full[32]
0,jf,16,16,21,7158,2677.297072,-5312.594145,-5168.198441,2026-07-27T11:19:45+00:00,0.208,...,0.856897,0.102093,1.310081,1.809776,2.009503,2.336942,2.517484,2.537755,2.672389,2.779141
1,kr,16,16,21,7023,3182.504234,-6323.008469,-6179.012608,2026-07-27T11:24:09+00:00,0.206,...,0.563427,0.000344,1.144367,1.774257,2.408940,2.814676,3.305561,3.772297,4.228134,4.709990
2,nh,16,16,21,7879,4967.571920,-9893.143841,-9746.732759,2026-07-27T11:27:23+00:00,0.226,...,1.476569,0.725912,1.672467,2.421798,3.001490,3.303553,3.559567,3.806757,3.736350,4.198793


In [19]:
fit_srdm = pd.read_csv("fits_srdm_nolapse.csv")
print(fit_srdm["log_lik_total"])
fit_srdm

0     3296.929346
1     3191.755550
2     5230.668048
3     3381.230583
4     3250.734942
5     5260.404265
6     3451.656173
7     3361.848034
8     5328.914743
9     3392.464804
10    5350.898179
11    3459.889393
12    3389.912606
13    5353.081184
14    3466.981602
15    3382.928202
16    5380.069097
17    3469.888565
18    3410.735610
19    5376.653831
20    3463.685739
21    3406.893190
22    5373.696126
23    3471.587956
24    3413.374977
25    5396.486034
26    3509.670292
27    3412.763164
28    5393.008357
29    3510.125094
30    3411.756709
31    5405.333786
32    3522.720923
33    3432.708515
34    5403.669783
35    3523.961533
36    3442.738301
37    5406.345186
Name: log_lik_total, dtype: float64


,pid,n_levels_requested,n_levels_actual,n_params,n_trials,log_lik_total,AIC,BIC,fit_time_utc,t0_hi,...,pf[45],pf[46],vk[45],vk[46],vf[45],vf[46],sigma_k[45],sigma_k[46],sigma_f[45],sigma_f[46]
0,jf,5,5,10,7179,3296.929346,-6573.858692,-6505.069538,2026-07-23T11:42:39+00:00,0.210,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,kr,5,5,10,7043,3191.755550,-6363.511100,-6294.913205,2026-07-23T11:42:56+00:00,0.208,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,nh,5,5,10,7904,5230.668048,-10441.336096,-10371.584854,2026-07-23T11:43:41+00:00,0.228,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,jf,7,7,12,7174,3381.230583,-6738.461165,-6655.922541,2026-07-23T11:44:16+00:00,0.210,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,kr,7,7,12,7039,3250.734942,-6477.469885,-6395.159228,2026-07-23T11:44:35+00:00,0.209,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,nh,7,7,12,7899,5260.404265,-10496.808530,-10413.114633,2026-07-23T11:45:23+00:00,0.226,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,jf,9,9,14,7167,3451.656173,-6875.312346,-6779.030952,2026-07-23T11:45:58+00:00,0.208,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,kr,9,9,14,7029,3361.848034,-6695.696069,-6599.686872,2026-07-23T11:46:18+00:00,0.206,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,nh,9,9,14,7892,5328.914743,-10629.829486,-10532.199018,2026-07-23T11:47:06+00:00,0.226,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,kr,10,10,15,7030,3392.464804,-6754.929607,-6652.060478,2026-07-23T11:39:23+00:00,0.208,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
